# 13. Production Error Handling for LangChain OpenAI

Build a reliable calling layer for missing API keys, authentication errors, rate limits, timeouts, connection failures, temporary server errors and invalid structured output.

## Failure categories

| Failure | Typical cause | Recommended response |
|---|---|---|
| Missing key | Environment not configured | Stop before calling the API |
| Authentication | Invalid or revoked key | Do not retry; request configuration correction |
| Rate limit | Too many requests or quota | Retry with bounded exponential backoff |
| Timeout | Slow network or model response | Retry a small number of times |
| Connection/server | Temporary infrastructure issue | Retry and then use a fallback |
| Invalid structure | Output does not match schema | Validate, optionally retry once, then fail safely |

### 1. Set up the API key and model

This cell imports the required classes, reads the API key securely, and selects the model without exposing credentials.

**Expected result:** No model output is produced; the environment becomes ready for later API calls. Read the output before continuing to the next cell.

In [ ]:
import os, getpass, time, random, logging
from langchain_openai import ChatOpenAI
from openai import AuthenticationError, RateLimitError, APITimeoutError, APIConnectionError, InternalServerError
from pydantic import BaseModel, Field, ValidationError

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
logger = logging.getLogger("llm_app")

### 2. Set up the API key and model

This cell imports the required classes, reads the API key securely, and selects the model without exposing credentials.

**Expected result:** No model output is produced; the environment becomes ready for later API calls. Read the output before continuing to the next cell.

In [ ]:
def require_api_key():
    key=os.getenv("OPENAI_API_KEY")
    if not key:
        key=getpass.getpass("Enter OPENAI_API_KEY: ").strip()
        if not key:
            raise RuntimeError("OPENAI_API_KEY is required. No API request was made.")
        os.environ["OPENAI_API_KEY"]=key
    return True

require_api_key()
MODEL_NAME=os.getenv("OPENAI_MODEL","gpt-4.1-mini")

### 3. Prepare the next processing step

This cell prepares the variables, functions, or validation logic required by the next stage of the example.

**Expected result:** The cell defines reusable objects or prints a small verification result. Read the output before continuing to the next cell.

In [ ]:
# Built-in retries cover many temporary failures.
llm=ChatOpenAI(
    model=MODEL_NAME,
    temperature=0,
    timeout=30,
    max_retries=2,
    max_completion_tokens=300
)

### 4. Handle errors and record operational status

This cell handles failures safely and records useful operational information without exposing secrets.

**Expected result:** Successful results are returned normally; known failures produce controlled fallback responses. Read the output before continuing to the next cell.

In [ ]:
def fallback_response(message="The AI service is temporarily unavailable."):
    return {"status":"fallback","answer":message,"retry_later":True}

def safe_invoke(prompt):
    try:
        response=llm.invoke(prompt)
        return {"status":"success","answer":response.content,"usage":response.usage_metadata}
    except AuthenticationError:
        logger.error("Authentication failed. Verify the configured API key.")
        return {"status":"configuration_error","answer":"Authentication failed.","retry_later":False}
    except RateLimitError:
        logger.warning("Rate limit reached after configured retries.")
        return fallback_response("The service is busy. Please try again shortly.")
    except APITimeoutError:
        logger.warning("The model request timed out.")
        return fallback_response("The request timed out. Please try again.")
    except (APIConnectionError, InternalServerError):
        logger.exception("Temporary connection or server failure.")
        return fallback_response()
    except Exception:
        logger.exception("Unexpected LLM failure.")
        return fallback_response("The request could not be completed safely.")

result=safe_invoke("Explain rate limiting in two simple sentences.")
print(result)

### 5. Handle errors and record operational status

This cell handles failures safely and records useful operational information without exposing secrets.

**Expected result:** Successful results are returned normally; known failures produce controlled fallback responses. Read the output before continuing to the next cell.

In [ ]:
# Explicit bounded backoff when application-level control is required.
def call_with_backoff(prompt, attempts=3, base_delay=1.0):
    for attempt in range(attempts):
        try:
            return llm.invoke(prompt).content
        except (RateLimitError, APITimeoutError, APIConnectionError, InternalServerError):
            if attempt==attempts-1:
                return fallback_response()
            delay=min(base_delay*(2**attempt)+random.uniform(0,0.25),5)
            logger.warning("Temporary failure; retrying in %.2f seconds",delay)
            time.sleep(delay)

print(call_with_backoff("Give one benefit of exponential backoff."))

### 6. Define and validate structured output

This cell defines a Pydantic schema and configures structured output so model results have predictable fields and types.

**Expected result:** A validated Python object or dictionary is returned instead of unstructured text. Read the output before continuing to the next cell.

In [ ]:
class TicketResult(BaseModel):
    category:str=Field(min_length=1)
    priority:str=Field(pattern="^(Low|Medium|High|Critical)$")
    summary:str=Field(min_length=1,max_length=200)

structured_llm=llm.with_structured_output(TicketResult)

def safe_structured_call(text):
    try:
        return {"status":"success","data":structured_llm.invoke(text).model_dump()}
    except (ValidationError, ValueError) as exc:
        logger.warning("Invalid structured output: %s",type(exc).__name__)
        return {"status":"invalid_output","data":None}
    except Exception:
        logger.exception("Structured request failed.")
        return {"status":"fallback","data":None}

print(safe_structured_call("Classify: Production checkout is unavailable for every customer."))

### 7. Prepare the next processing step

This cell prepares the variables, functions, or validation logic required by the next stage of the example.

**Expected result:** The cell defines reusable objects or prints a small verification result. Read the output before continuing to the next cell.

In [ ]:
# Process a batch without losing successful rows when one request fails.
records=["Login reset required","Duplicate invoice charge","Production API unavailable"]
batch_results=[]
for record_id,text in enumerate(records,1):
    item=safe_invoke(f"Classify this support request in one short line: {text}")
    batch_results.append({"record_id":record_id,"input":text,"status":item["status"],"output":item["answer"]})

batch_results

### 8. Handle errors and record operational status

This cell handles failures safely and records useful operational information without exposing secrets.

**Expected result:** Successful results are returned normally; known failures produce controlled fallback responses. Read the output before continuing to the next cell.

In [ ]:
# Safe logging: record operational metadata, not secrets or raw sensitive text.
for item in batch_results:
    logger.info("record_id=%s status=%s output_length=%s",item["record_id"],item["status"],len(str(item["output"])))

summary={"total":len(batch_results),"successful":sum(x["status"]=="success" for x in batch_results)}
print(summary)

## Production checklist

- Keep keys in a secret manager or environment variable; never print them.
- Do not retry authentication or validation errors blindly.
- Use bounded retries with exponential backoff and jitter.
- Set request timeouts and concurrency limits.
- Validate structured output before downstream use.
- Return a clear, non-technical fallback message.
- Log request IDs, status, latency and error category without sensitive input.
- Monitor failure rates and alert when thresholds are exceeded.

## Testing the failure paths

Use mocks in automated tests to raise each exception deliberately. Confirm that authentication errors stop immediately, temporary errors retry only within the limit, invalid output never reaches downstream systems, logs contain no secrets, and every exhausted path returns the documented fallback structure.